# Praktikum Kecerdasan Artifisial Lanjut


---
## Bab 6. Support Vector Machine (SVM)

Nama: Erza Hanif Pramudita Hanggara

NIM: 245150200111038

Kelas: TIF-A

### Percobaan Import Data

Unduh dataset yang akan digunakan pada praktikum kali ini. Anda dapat menggunakan aplikasi wget untuk mendowload dataset dan menyimpannya dalam Google Colab. Jalankan cell di bawah ini untuk mengunduh dataset

In [509]:
!wget https://gist.githubusercontent.com/Thanatoz-1/9e7fdfb8189f0cdf5d73a494e4a6392a/raw/aaecbd14aeaa468cd749528f291aa8a30c2ea09e/iris_dataset.csv

--2026-04-02 16:20:44--  https://gist.githubusercontent.com/Thanatoz-1/9e7fdfb8189f0cdf5d73a494e4a6392a/raw/aaecbd14aeaa468cd749528f291aa8a30c2ea09e/iris_dataset.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4626 (4.5K) [text/plain]
Saving to: ‘iris_dataset.csv.19’

iris_dataset.csv.19 100%[===================>]   4.52K  --.-KB/s    in 0s      

2026-04-02 16:20:44 (55.2 MB/s) - ‘iris_dataset.csv.19’ saved [4626/4626]



Setelah dataset berhasil diunduh, langkah berikutnya adalah membaca dataset dengan memanfaatkan fungsi **readcsv** dari library pandas. Lakukan pembacaan berkas csv ke dalam dataframe dengan nama **data** menggunakan fungsi **readcsv**. Jangan lupa untuk melakukan import library pandas terlebih dahulu


In [510]:
import pandas as pd
import numpy as np
data = pd.read_csv('iris_dataset.csv')



Cek isi dataset Anda dengan menggunakan perintah **head()**

In [511]:
data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### **6.4.2 Klasifikasi Biner**

Karena metode dasar Support Vector Machine (SVM) hanya mampu melakukan klasifikasi data biner (dua kelas), sedangkan dataset Iris memiliki tiga kelas, maka perlu dilakukan penghapusan salah satu kelas. Dalam percobaan ini, kelas yang dihapus adalah Iris-virginica.

Berikut adalah kode untuk menghapus kelas Iris-virginica dari dataset:

In [512]:
data.drop(data[data['target'] == 'Iris-virginica'].index, inplace=True)

SVM hanya dapat menerima kelas dalam bentuk numerik. Oleh karena itu, perlu dilakukan konversi label kelas menjadi angka. Kode berikut mengubah label kelas Iris-setosa menjadi -1 dan Iris-versicolor menjadi 1:

In [513]:
data['target']=data['target'].map({'Iris-setosa':-1,'Iris-versicolor':1})

Terakhir, beberapa baris data ditampilkan untuk memverifikasi perubahan:

In [514]:
data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,-1
1,4.9,3.0,1.4,0.2,-1
2,4.7,3.2,1.3,0.2,-1
3,4.6,3.1,1.5,0.2,-1
4,5.0,3.6,1.4,0.2,-1


#### 1) Percobaan Membagi Data

Metode pembelajaran mesin memerlukan dua jenis data :


1.   Data latih : Digunakan untuk proses training metode klasifikasi
2.   Data uji : Digunakan untuk proses evaluasi metode klasifikasi

Data uji dan data latih perlu dibuat terpisah (mutualy exclusive) agar hasil evaluasi lebih akurat.

Data uji dan data latih dapat dibuat dengan cara membagi dataset dengan rasio tertentu, misalnya 80% data latih dan 20% data uji.

Library Scikit-learn memiliki fungsi [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) pada modul **model_selection** untuk membagi dataset menjadi data latih dan data uji. Bagilah dataset anda menjadi dua, yaitu **data_latih** dan **data_uji**.


In [515]:
from sklearn.model_selection import train_test_split
data_latih, data_uji = train_test_split(data, test_size=0.2)

Tampilkan banyaknya data pada **data_latih** dan **data_uji**. Seharusnya **data_latih** terdiri dari 80 data, dan **data_uji** terdiri dari 20 data

In [516]:
print(data_uji.shape[0])
print(data_latih.shape[0])

20
80


Pisahkan label/kelas dari data uji menjadi sebuah variabel bernama **label_uji**

In [517]:
label_latih = data_latih.pop('target')
label_uji = data_uji.pop('target')

#### 2) Percobaan Proses Training

Tujuan dari algoritma SVM adalah meminimalkan nilai cost function. Penghitungan nilai minimal dapat dapat dilakukan dengan menghitung nilai gradient dari cost function terlebih dahulu. Fungsi di bawah ini berguna untuk menghitung nilai gradient cost function seperti yang tertera pada bagian teori.

In [518]:
def hitung_cost_gradient(W, X, Y, regularization):
  jarak = 1 - (Y * np.dot(X, W))
  dw = np.zeros(len(W))
  if max(0, jarak) == 0:
    di = W
  else:
    di = W - (regularization * Y * X)
  dw += di
  return dw

Terdapat beberapa cara untuk meminimalkan nilai cost function, salah satunya menggunakan Stochastic Gradient Descent (SGD) untuk melakukan minimasi. Minimasi cost function merupakan inti dari algoritma SVM. Fungsi di bawah ini merupakan implementasi algoritma SGD:

In [519]:
from sklearn.utils import shuffle
def sgd(data_latih, label_latih, learning_rate = 0.000001,
        max_epoch = 1000, regularization = 10000):
  data_latih = data_latih.to_numpy()
  label_latih = label_latih.to_numpy()
  bobot = np.zeros(data_latih.shape[1])
  for epoch in range(1, max_epoch):
    X, Y = shuffle(data_latih, label_latih, random_state = 101)
    for index, x in enumerate(X):
      delta = hitung_cost_gradient(bobot, x, Y[index], regularization)
      bobot = bobot - (learning_rate * delta)
  return bobot

Pada algoritma SGD, dilakukan perulangan sebanyak max_epoch dan dilakukan perhitungan bobot, yang merupakan parameter hyperplane. Pada setiap perulangan dilakukan pengacakan data, agar tidak terjadi pola perhitungan bobot yang sama. Nilai bobot diperoleh dari bobot dikurangi gradien dikalikan dengan learning rate. Nilai learning rate merupakan sebuah parameter yang mengatur seberapa besar perubahan bobot.

In [520]:
W = sgd(data_latih, label_latih)
print(W)

[-0.22336285 -0.61786015  0.99091025  0.42322425]


#### 3) Percobaan Proses Testing
Proses testing dilakukan dengan menghitung nilai [*dot product*](https://en.wikipedia.org/wiki/Dot_product) antara bobot hasil training dengan data uji. Kelas data ditentukan berdasarkan tanda (positif atau negatif) dari hasil dot product tersebut. Fungsi berikut mengimplementasikan proses testing

In [521]:
def testing(W, data_uji):
  prediksi = np.array([])
  for i in range(data_uji.shape[0]):
    y_prediksi = np.sign(np.dot(W, data_uji.to_numpy()[i]))
    prediksi = np.append(prediksi, y_prediksi)
  return prediksi

Lakukan pengujian menggunakan seluruh data uji. Bandingkan hasilnya dengan nilai label sebenernya untuk menghitung berapa banyak data uji yang berhasil diprediksi dengan benar.

In [522]:
y_prediksi = testing(W, data_uji)
print(sum(y_prediksi==label_uji))

20


### **6.4.3 Klasifikasi Multi-Class**

Masukkan kembali dataset ke sebuah dataframe dengan nama yang berbeda dari klasifikasi biner.

In [523]:
df = pd.read_csv('iris_dataset.csv')

In [524]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.2)

print(df_train.shape[0])
print(df_test.shape[0])

120
30


In [525]:
label_train = df_train.pop('target')
label_test = df_test.pop('target')

#### 1) Percobaan Pembuatan Data Latih One-vs-rest

Metode one-vs-rest memerlukan tiga jenis data latih yang diperlukan untuk melatih tiga SVM yang berbeda pada dataset Iris. Fungsi **buat_trainingset** digunakan untuk membentuk tiga dataset tersebut.

In [526]:
def buat_trainingset(dataset):
  trainingset = {}
  # Asumsi kolom target adalah kolom terakhir
  target_column_name = dataset.columns[-1]
  # Ambil nilai unik dari kolom target (Series)
  list_kelas = dataset[target_column_name].unique()
  for kelas in list_kelas:
    data_temp = dataset.copy(deep=True)
    # Lakukan mapping pada kolom target
    data_temp[target_column_name]=data_temp[target_column_name].map({kelas:1})
    # Isi kelas lain dengan -1
    data_temp[target_column_name]=data_temp[target_column_name].fillna(-1)
    trainingset[kelas]=data_temp
  return trainingset

Gunakan fungsi **buat_trainingset** untuk membentuk data latih dengan nama variabel **trainingset** yang akan digunakan pada proses training.

In [527]:
trainingset = buat_trainingset(data_latih)

Tampilkan isi **trainingset** agar Anda dapat memahami struktur dari variabel tersebut.

In [528]:
print(trainingset)

{np.float64(1.4):     sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
75                6.6               3.0                4.4               1.0
50                7.0               3.2                4.7               1.0
21                5.1               3.7                1.5              -1.0
89                5.5               2.5                4.0              -1.0
48                5.3               3.7                1.5              -1.0
..                ...               ...                ...               ...
12                4.8               3.0                1.4              -1.0
39                5.1               3.4                1.5              -1.0
49                5.0               3.3                1.4              -1.0
25                5.0               3.0                1.6              -1.0
54                6.5               2.8                4.6              -1.0

[80 rows x 4 columns], np.float64(0.4):     sepal length 

In [529]:
trainingset_df = df_train.copy()
trainingset_df['target'] = label_train
trainingset = buat_trainingset(trainingset_df)

#### 2) Percobaan Proses Training

Proses training dilakukan dengan memanggil fungsi **sgd** berulang kali sesuai banyaknya kelas yang ada pada data. Dengan demikian, proses training menghasilkan bobot sebanyak kelas yang ada pada dataset. Buatlah fungsi bernama **training** yang digunakan untuk melakukan proses training one-vs-rest

In [530]:
def training(trainingset):
  list_kelas = trainingset.keys()
  W = {}
  for kelas in list_kelas:
    data_latih = trainingset[kelas]
    label_latih = data_latih.pop(data_latih.columns[-1])
    W[kelas] = sgd(data_latih, label_latih)
  return W

Lakukan proses training dengan memanggil fungsi **training** dan menempatkan hasilnya pada variabel **W**

In [531]:
W = training(trainingset)

Tampilkan isi variabel **W**

In [532]:
W

{'Iris-virginica': array([-4.04177956, -4.0904806 ,  4.95018135,  7.2155169 ]),
 'Iris-versicolor': array([ 0.75627417, -1.80045039,  1.26769819, -3.41252478]),
 'Iris-setosa': array([ 0.15981481,  0.66834083, -0.94976498, -0.4247419 ])}

#### 3) Percobaan Proses Testing

Proses testing dilakukan dengan menghitung nilai dot product antara bobot hasil training dengan data uji. Kelas data ditentukan berdasarkan tanda (positif atau negatif) dari hasil dot product tersebut. Fungsi berikut mengimplementasikan proses *testing*.

In [533]:
def testing(W, data_uji):
  prediksi = np.array([])
  for i in range(data_uji.shape[0]):
    y_prediksi = np.sign(np.dot(W, data_uji.to_numpy()[i]))
    prediksi = np.append(prediksi, y_prediksi)
  return prediksi

In [534]:
def testing_onevsrest(W, data_uji):
  kelas_prediksi = []

  # Convert data_uji DataFrame to numpy array for easier row iteration
  data_uji_np = data_uji.to_numpy()

  for x in data_uji_np:
    # W's weights will now be trained on 4 features, so use all features from x
    x_features = x # Use all 4 features from the test data point

    scores = {}
    for kelas, weights in W.items():
      scores[kelas] = np.dot(weights, x_features) # Dot product between (4,) and (4,)

    # Determine the predicted class based on the highest score
    # This is a common strategy for One-vs-Rest classification
    predicted_kelas = max(scores, key=scores.get)
    kelas_prediksi.append(predicted_kelas)

  return kelas_prediksi

Lakukan pengujian menggunakan seluruh data uji. Bandingkan hasilnya dengan nilai label sebenarnya untuk menghitung berapa banyak data uji yang berhasil diprediksi dengan benar.

In [535]:
y_prediksi = testing_onevsrest(W, df_test)
print(sum(y_prediksi==label_test))

29


## TUGAS
Pada tugas kali ini Anda mendefinisikan proses testing pada metode one-vs-rest. Proses testing pada metode one-vs-rest dilakukan dengan memanggil proses testing biner untuk setiap **value** pada dictionary **W**. Kelas pada sebuah data latih adalah **key** pada dictionary **W** yang memiliki nilai prediksi **1**. Lengkapi fungsi **testing_onevsrest** di bawah ini. Output dari fungsi tersebut adalah list nama kelas hasil prediksi.

In [536]:
def testing_onevsrest(W, data_uji):
  kelas_prediksi = []

  # Convert data_uji DataFrame to numpy array for easier row iteration
  data_uji_np = data_uji.to_numpy()

  for x in data_uji_np:
    # W's weights will now be trained on 4 features, so use all features from x
    x_features = x # Use all 4 features from the test data point

    scores = {}
    for kelas, weights in W.items():
      scores[kelas] = np.dot(weights, x_features) # Dot product between (4,) and (4,)

    # Determine the predicted class based on the highest score
    # This is a common strategy for One-vs-Rest classification
    predicted_kelas = max(scores, key=scores.get)
    kelas_prediksi.append(predicted_kelas)

  return kelas_prediksi

In [537]:
prediksi = testing_onevsrest(W, df_test)

Berapa banyak data latih yang berhasil diprediksi dengan benar?

In [538]:
print(sum(prediksi == label_test))

29


## KESIMPULAN

Di praktikum kali ini kita sudah:
- Mempraktikkan metode SVM untuk klasifikasi dataset dengan 3 kelas
- Belajar bahwa SVM sebenarnya hanya bisa menangani klasifikasi biner (2 kelas)
- Menggunakan strategi One-vs-Rest untuk mengakali masalah ini

Buatlah kesimpulan apa yang sudah dipelajari di praktikum dan penugasan ini. Kesimpulan harus berisi:
- Apakah berhasil dalam mengimplementasikan SVM multi-kelas?
- Seberapa akurat hasil klasifikasinya?
- Adakah temuan yang menarik atau tidak terduga?
- Apa saja hambatan yang ditemui selama praktikum?
- Bagaimana cara mengatasinya?

- Ya, berhasil mengimplementasikan SVM untuk klasifikasi multi-kelas menggunakan strategi One-vs-Rest.
- Hasil klasifikasi menunjukkan akurasi yang tinggi, dengan 29 dari 30 data uji berhasil
- Temuan menariknya adalah meskipun SVM secara inheren adalah pengklasifikasi biner, pendekatan One-vs-Rest memungkinkan perluasan fungsionalitasnya untuk menangani masalah multi-kelas secara efektif
- Hambatannya adalah dari testing pada multi class svm, dimana dia tidak bisa dijalankan dan muncul error
- Solusi yang dapat digunakan adalah menggunakan method testing onevsrest yang didefinisikan di tugas praktikum
